## Random Forest

In [ ]:
X1, X2, X3, X4, X5 = create_kfold_sets(data_train=data_train) #Model without feature selection
#X1, X2, X3, X4, X5 = create_kfold_sets(data_train=data_train_wrapper) #Model with feature selection

D = [X1, X2, X3, X4, X5]

In [ ]:
# Define the columns for the results DataFrame
columns = ['Model', 'N_estimators', 'Criterion', 'Min_samples_leaf',
           'Accuracy',
           'Recall',
           'Specificity',
           'Precision',
           'F1']
df_results = pd.DataFrame(columns=columns)

# Define parameters for Grid Search
n_estimators = [25, 50, 75, 100]
criterion = ['gini', 'entropy']
min_samples_leaf = [1, 2, 4, 8, 16, 32]
model_name = 'Random Forest'

for i in n_estimators:
    for k in criterion:
        for d in min_samples_leaf:
            columns_results=['Model', 'Accuracy', 'Recall', 'Specificity', 'Precision', 'F1']
            df_results_fold = pd.DataFrame(columns=columns_results)
            for j in range(5):
                # Prepare test and train sets for this fold
                d_test = pd.concat([D[j]])
                y_test = d_test['label_binary']
                X_test = d_test.drop(columns=['label_binary', 'n_image', 'label_multi'])
                d_train = pd.concat([D[k] for k in range(5) if k != j], ignore_index=True)
                y_train = d_train['label_binary']
                X_train = d_train.drop(columns=['label_binary', 'n_image', 'label_multi'])

                # Initialize and fit the model
                model = RandomForestClassifier(
                    n_estimators=i,
                    criterion=k,
                    min_samples_leaf=d,
                    bootstrap=True,
                    random_state=42)
                model.fit(X_train, y_train)

                y_pred = model.predict(X_test)

                print(f'Model with {i} estimators:')
                results=evaluate_model(y_pred, y_test, model=model_name)
                df_results_fold = pd.concat([df_results_fold, results], ignore_index=True)
                print('\n')

            # Calculate mean metrics across folds
            recall_mean, specificity_mean, precision_mean, f1_mean, accuracy_mean = means_results(df_results_fold)
            # Append results to DataFrame
            result_i = {'Model': model_name, 'Accuracy': accuracy_mean, 'N_estimators': i, 'Criterion': k, 'Min_samples_leaf': d,
                        'Recall': recall_mean, 'Specificity': specificity_mean, 'Precision': precision_mean, 'F1': f1_mean}
            df_results = pd.concat([df_results, pd.DataFrame([result_i])], ignore_index=True)


In [ ]:
df_results.sort_values(by='Recall', ascending=False)

Chosen model

In [ ]:
# Parameters
min_samples = 1
n_estimators = 50
criterion = 'entropy'

# Assuming data_train and data_test are already defined
X_train = data_train.drop(columns=['label_binary', 'n_image', 'label_multi'])
y_train = data_train['label_binary']
X_test = data_test.drop(columns=['label_binary', 'n_image', 'label_multi'])
y_test = data_test['label_binary']

# Initialize and train the model
model = RandomForestClassifier(
    n_estimators=n_estimators,
    criterion=criterion,
    min_samples_leaf=min_samples,
    bootstrap=True,
    random_state=42
)
model.fit(X_train, y_train)

# Measure execution time
t0 = time.time()
y_pred = model.predict(X_test)
t1 = time.time()

df_results = pd.DataFrame(columns=['Model', 'Accuracy', 'Recall', 'Specificity', 'Precision', 'F1'])
results=evaluate_model(y_pred, y_test, model=model_name)
df_results = pd.concat([df_results, results], ignore_index=True)
time_taken = t1 - t0
time_taken = round(time_taken, 3)
print(f'Execution time: {time_taken} seconds')


In [ ]:
# Get feature importances from the trained model
importancias = model.feature_importances_

# Plot feature importances as a bar chart
plt.figure(figsize=(10, 6))
plt.bar(X_train.columns, importancias, color='skyblue')
plt.xticks(rotation=90, fontsize=10)
plt.xlabel('Features', fontsize=12)
plt.ylabel('Importance', fontsize=12)
plt.title('Feature Importances from Random Forest', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# Create a DataFrame from feature importances
importancias_df = pd.DataFrame(importancias, index=X_train.columns, columns=['Importance'])

# Sort the DataFrame by importance in descending order
sorted_importances_df = importancias_df.sort_values(by='Importance', ascending=False)
print(sorted_importances_df)
